In [5]:
import cv2
import glob
from natsort import natsorted

# Путь к PNG
image_files = natsorted(glob.glob("./lightning_logs/GAN/v10_v10_pic_file_per_batch/img/*.png"))

# Берем размеры из первого кадра
frame = cv2.imread(image_files[0])
h, w, _ = frame.shape

# Создаем видео
out = cv2.VideoWriter('training.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 30, (w, h))  # 30 fps

for n, img_path in enumerate(image_files):
    print(f"Processing frame {n} of {len(image_files)} ({100 * n / len(image_files)}%)")
    frame = cv2.imread(img_path)
    out.write(frame)

out.release()
print("Видео training.mp4 готово")


Processing frame 0 of 2325 (0.0%)
Processing frame 1 of 2325 (0.043010752688172046%)
Processing frame 2 of 2325 (0.08602150537634409%)
Processing frame 3 of 2325 (0.12903225806451613%)
Processing frame 4 of 2325 (0.17204301075268819%)
Processing frame 5 of 2325 (0.21505376344086022%)
Processing frame 6 of 2325 (0.25806451612903225%)
Processing frame 7 of 2325 (0.3010752688172043%)
Processing frame 8 of 2325 (0.34408602150537637%)
Processing frame 9 of 2325 (0.3870967741935484%)
Processing frame 10 of 2325 (0.43010752688172044%)
Processing frame 11 of 2325 (0.4731182795698925%)
Processing frame 12 of 2325 (0.5161290322580645%)
Processing frame 13 of 2325 (0.5591397849462365%)
Processing frame 14 of 2325 (0.6021505376344086%)
Processing frame 15 of 2325 (0.6451612903225806%)
Processing frame 16 of 2325 (0.6881720430107527%)
Processing frame 17 of 2325 (0.7311827956989247%)
Processing frame 18 of 2325 (0.7741935483870968%)
Processing frame 19 of 2325 (0.8172043010752689%)
Processing frame

In [ ]:
import cv2
import glob
from natsort import natsorted

# Путь к PNG
image_files = natsorted(glob.glob("./lightning_logs/WGAN/gen_no_instNorm_no_tail_relu/img/*.png"))

# Берем размеры из первого кадра
frame = cv2.imread(image_files[0])
h, w, _ = frame.shape

# Создаем видео
out = cv2.VideoWriter('training.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 30, (w, h))  # 30 fps

for n, img_path in enumerate(image_files):
    print(f"Processing frame {n} of {len(image_files)} ({100 * n / len(image_files)}%)")
    frame = cv2.imread(img_path)
    out.write(frame)

out.release()
print("Видео training.mp4 готово")

In [ ]:
import cv2
import glob
import numpy as np
from natsort import natsorted

# ---------------- config ----------------
IMAGE_GLOB = "./lightning_logs/WGAN/gen_no_instNorm_no_tail_relu/img/*.png"
OUTPUT_VIDEO = "training_pingpong.mp4"

FPS = 30
INTERP_FRAMES = 3   # доп. кадры между keyframes
# ----------------------------------------

image_files = natsorted(glob.glob(IMAGE_GLOB))
assert len(image_files) > 1

def crop_bottom_right(img):
    """
    ЧЕТКИЙ кроп: 4-й по горизонтали, 2-й по вертикали
    без залезания на соседние
    """
    h, w, _ = img.shape
    tile_w = w // 4
    tile_h = h // 2

    x0 = 3 * tile_w
    y0 = 1 * tile_h

    return img[y0:y0 + tile_h, x0:x0 + tile_w]

def interpolate(a, b, n):
    """Линейная интерполяция кадров"""
    frames = []
    for i in range(1, n + 1):
        alpha = i / (n + 1)
        frames.append(
            cv2.addWeighted(a, 1 - alpha, b, alpha, 0)
        )
    return frames

# --------- собираем все кадры ----------
frames = []

prev = cv2.imread(image_files[0])
# prev = crop_bottom_right(prev)
frames.append(prev)

for path in image_files[1:]:
    curr = cv2.imread(path)
    # curr = crop_bottom_right()
    frames.extend(interpolate(prev, curr, INTERP_FRAMES))
    frames.append(curr)
    prev = curr

# --------- ping-pong (без дублей концов) ----------
pingpong_frames = frames + frames[-2:0:-1]

# --------- видео ----------
h, w, _ = pingpong_frames[0].shape
out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    FPS,
    (w, h)
)

for i, f in enumerate(pingpong_frames):
    out.write(f)
    if i % 50 == 0:
        print(f"{i}/{len(pingpong_frames)}")

out.release()
print(f"Видео {OUTPUT_VIDEO} готово")


error: OpenCV(4.12.0) /io/opencv/modules/core/src/arithm.cpp:662: error: (-209:Sizes of input arguments do not match) The operation is neither 'array op array' (where arrays have the same size and the same number of channels), nor 'array op scalar', nor 'scalar op array' in function 'arithm_op'
